<a href="https://colab.research.google.com/github/Akshaya-20012604/LeetCodeSolutions/blob/main/travel_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-adk google-generativeai -q

In [ ]:
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
import vertexai
from google.colab import auth
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")

✅ All libraries are ready to go!


In [ ]:
# ---  Authentication & Project Configuration ---

# Authenticate user in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Authenticated successfully.")

✅ Authenticated successfully.


In [ ]:
# @title Set Your Google Cloud Project Details
PROJECT_ID = "mercurial-feat-490607-p7"             # @param {type:"string"}
LOCATION = "us-central1"               # @param {type:"string"}

# Set environment variables for the ADK and gcloud
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

# Initialize Vertex AI with the specified project and location
vertexai.init(project=PROJECT_ID, location=LOCATION)

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")


✅ Vertex AI configured for project 'mercurial-feat-490607-p7' in 'us-central1'.


In [ ]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating Total Day spontaneous itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Trip" Generator 🚗 - a specialized AI assistant that creates engaging itineraries for the Total days.

        Your Mission:
        Transform a simple mood or interest into a complete adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Total Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names along with ticket details.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [ ]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [ ]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a trip from Pune to Delhi with total days intineraries, Give me all the budget details including flight costs, train costs, also include flights available on 10th April 2026 with flight timings also list available hotels in delhi to stay with thier ratings and cost "
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a trip from Pune to Delhi with total days intineraries, Give me all the budget details including flight costs, train costs, also include flights available on 10th April 2026 with flight timings also list available hotels in delhi to stay with thier ratings and cost '

🚀 Running query for agent: 'day_trip_agent' in session: '865e1e4e-0bdc-4423-ba69-fddb9e4ed410'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Planning a spontaneous trip from Pune to Delhi requires careful consideration of travel costs, accommodation, and an engaging itinerary. Given that April 10, 2026, is very soon, flight and train ticket prices may be higher, and availability might be limited.

I will plan a 3-day itinerary for your visit to Delhi, assuming a moderate budget for activities.

---

### **Pune to Delhi Travel Details (April 10, 2026)**

#### **Flight Options**

Booking flights two days in advance for April 10, 2026, can result in higher 

Planning a spontaneous trip from Pune to Delhi requires careful consideration of travel costs, accommodation, and an engaging itinerary. Given that April 10, 2026, is very soon, flight and train ticket prices may be higher, and availability might be limited.

I will plan a 3-day itinerary for your visit to Delhi, assuming a moderate budget for activities.

---

### **Pune to Delhi Travel Details (April 10, 2026)**

#### **Flight Options**

Booking flights two days in advance for April 10, 2026, can result in higher prices. However, here are some potential options identified:

*   **Air India:** Prices can start from approximately ₹3,800 (US$46) for a one-way ticket on April 10, 2026.
    *   One flight option mentioned for April 10, 2026: **PNQ 22:00 (Pune) - DEL 00:15 (Delhi)** (next day arrival).
*   **Other Airlines (IndiGo, Akasa Air, SpiceJet, Air India Express):** One-way flights from Pune to Delhi typically range from ₹3,894 to ₹15,400, depending on the airline and how far in advance you book. Average fares for April 2026 are around ₹7,200.
    *   **Average Flight Duration:** Approximately 2 hours 10 minutes to 2 hours 20 minutes for non-stop flights.

**Estimated Flight Cost (One-way, April 10, 2026):** ₹3,800 - ₹7,500+ depending on airline and booking time.

#### **Train Options**

Traveling by train from Pune to Delhi is a more budget-friendly option, though it takes significantly longer. Train ticket prices vary greatly by class and train type.

*   **Cheapest Option:** Jhelum Express (11077) has Sleeper (SL) class tickets starting from ₹695 and takes around 28 hours. It departs Pune at 17:20 and arrives in Delhi at 21:20 the next day.
*   **Fastest Option:** NZM Duronto (12263) takes about 19 hours 50 minutes. It departs Pune at 11:10 AM and arrives in Delhi at 07:00 AM the next day.
    *   **NZM Duronto Fares:** 3A (AC 3-tier) can be around ₹3,265.
*   **Other Trains:** Goa Express (12779) offers Sleeper class at ₹725 and 3E (AC Economy) at ₹1,730. It departs Pune at 04:30 AM and reaches Delhi at 06:25 AM the next day, taking approximately 25 hours 55 minutes.

**Estimated Train Cost (One-way):** ₹695 (Sleeper Class) to ₹5,475 (AC First Class).

---

### **Accommodation in Delhi**

Delhi offers a wide range of hotels across various budget categories. Here are a few options with their approximate costs and ratings:

#### **Budget-Friendly Hotels (Approx. ₹1,200 - ₹2,500 per night)**

*   **The Hosteller Delhi:** Located in South Delhi, rated 8.6/10 (Excellent), averaging around US$22 (~₹1,800) per night.
*   **Hotel BB Palace:** In Karol Bagh, rated 6.7/10 (Good), averaging around US$25 (~₹2,100) per night.
*   **Roomshala Hotel Anmol:** Budget hotel with prices starting from ₹446 per night.
*   **Hotel Vanson Villa, City Centre:** Offers good value, clean rooms, and is conveniently located.

#### **Mid-Range Hotels (Approx. ₹4,000 - ₹9,500 per night)**

*   **Hotel Palace Heights:** Centrally located in Connaught Place, ideal for tourists. It also features an excellent restaurant. Many mid-range hotels in Delhi are available for under US$120 (~₹10,000) a night.
*   **Alpina Hotels & Suites:** A good option in South Delhi with clean, spacious, air-conditioned rooms and a multi-cuisine restaurant.
*   **Bloomrooms @ Janpath:** Good option for central Delhi.

#### **Luxury Hotels (Approx. ₹10,000+ per night)**

*   **The Leela Palace New Delhi:** A 5-star hotel known for its opulent architecture and service, located in the Diplomatic Enclave. Prices from around €147 (~₹13,000) per night.
*   **The Imperial New Delhi:** A highly-rated 5-star luxury hotel.
*   **Taj Mahal, New Delhi:** An iconic hotel in Lutyens' Delhi known for its hospitality and modern facilities. Prices from around €126 (~₹11,000) per night.
*   **The Oberoi, New Delhi:** Offers contemporary luxury with views of the Delhi Golf Course and Humayun's Tomb. Prices from around €184 (~₹16,500) per night.

---

### **3-Day Delhi Itinerary**

This itinerary is designed for a general tourist interest, mixing historical, cultural, and local experiences. Please check current opening hours and ticket prices closer to your travel date.

**Day 1: Old Delhi Charm & Historical Grandeur**

*   **Morning (9:00 AM - 1:00 PM): Old Delhi Exploration**
    *   Arrive at **Red Fort**: Explore this UNESCO World Heritage site, built by Shah Jahan. (Entry Fee: approx. ₹500 for foreigners, ₹90 for Indians).
    *   Visit **Jama Masjid**: India's largest mosque, close to the Red Fort. (Entry Fee: Free, Camera Fee: approx. ₹300).
    *   **Chandni Chowk**: Wander through the bustling lanes of this historic market, famous for street food, textiles, and electronics.
*   **Lunch (1:00 PM - 2:00 PM):** Enjoy local delicacies at Chandni Chowk (e.g., Parathe Wali Gali, Karim's).
*   **Afternoon (2:00 PM - 6:00 PM): Rajpath & India Gate**
    *   Visit **India Gate**: A war memorial and iconic landmark.
    *   Drive past **Rashtrapati Bhavan** (President's House) and Parliament House.
    *   Explore **Raj Ghat**: The memorial to Mahatma Gandhi.
*   **Evening (6:00 PM onwards): Connaught Place**
    *   Stroll around **Connaught Place**: A colonial-era shopping and dining hub. Enjoy dinner at one of its many restaurants.

**Day 2: Mughal Marvels & Spiritual Serenity**

*   **Morning (9:00 AM - 1:00 PM): UNESCO World Heritage Sites**
    *   **Humayun's Tomb**: A magnificent Mughal edifice and a precursor to the Taj Mahal. (Entry Fee: approx. ₹500 for foreigners, ₹30 for Indians).
    *   **Lodhi Garden**: A beautiful garden housing tombs of Sayyid and Lodi rulers.
*   **Lunch (1:00 PM - 2:00 PM):** Lunch at a cafe near Lodhi Garden or Khan Market.
*   **Afternoon (2:00 PM - 6:00 PM): Tower of Victory & Stepwell**
    *   **Qutub Minar Complex**: Explore the Qutub Minar, Iron Pillar, and ruins of mosques. (Entry Fee: approx. ₹500 for foreigners, ₹30 for Indians).
    *   **Agrasen ki Baoli**: An ancient stepwell providing a peaceful escape.
*   **Evening (6:00 PM onwards): Spiritual & Cultural Experience**
    *   **Lotus Temple (Baháʼí House of Worship):** Admire its unique lotus-like architecture. (Entry Fee: Free).
    *   **Akshardham Temple:** If time permits, visit this grand Swaminarayan temple known for its intricate carvings and exhibitions. (Entry Fee: Free for temple, exhibitions extra).

**Day 3: Arts, Crafts & Leisure**

*   **Morning (9:00 AM - 1:00 PM): Science & Art**
    *   **Jantar Mantar**: An 18th-century astronomical observatory. (Entry Fee: approx. ₹200 for foreigners, ₹25 for Indians).
    *   **National Museum:** Explore India's rich history through its vast collection of art and artifacts.
*   **Lunch (1:00 PM - 2:00 PM):** Grab lunch at a local eatery.
*   **Afternoon (2:00 PM - 6:00 PM): Shopping & Local Flavor**
    *   **Dilli Haat**: Experience diverse Indian culture, crafts, and cuisine from various states.
    *   Alternatively, explore **Hauz Khas Village**: Known for its historical ruins, art galleries, cafes, and boutiques.
*   **Evening (6:00 PM onwards): Departure or Leisure**
    *   Enjoy a final dinner in Delhi, perhaps trying a different cuisine or revisiting a favorite spot. Depending on your departure schedule, head to the airport or railway station.

---

**Total Estimated Budget Considerations (per person, excluding personal shopping and miscellaneous expenses):**

*   **Flights:** ₹3,800 - ₹7,500 (one-way)
*   **Trains:** ₹695 - ₹5,475 (one-way, depending on class)
*   **Accommodation (3 nights):**
    *   Budget: ₹3,600 - ₹7,500
    *   Mid-Range: ₹12,000 - ₹28,500
    *   Luxury: ₹30,000 - ₹50,000+
*   **Daily Activities/Food/Local Transport (Estimated Moderate Budget):** ₹2,000 - ₹3,000 per day x 3 days = ₹6,000 - ₹9,000.

**Overall Trip Estimate (per person for 3 days):**
*   **Budget Traveler:** ₹10,000 - ₹20,000 (using cheapest train and budget hotel)
*   **Mid-Range Traveler:** ₹20,000 - ₹40,000 (using mid-range train/flight and mid-range hotel)
*   **Luxury Traveler:** ₹45,000 - ₹70,000+ (using flight and luxury hotel)

Remember to book flights and accommodation as soon as possible due to the proximity of the travel date. Enjoy your spontaneous trip to Delhi!

--------------------------------------------------

